# 03 — Building Characteristics (HCAD / PLUTO)

Aggregates building-level statistics per grid cell.

**Data sources:**
- HCAD (Houston): `hcad_path` if `needs_hcad=true`
- PLUTO (NYC): `pluto_path` if `needs_pluto=true`

**Method:** Load data, spatially assign to grid cells (matching notebook 01), aggregate per cell.

**Output columns:** `cell_id`, `avg_floors`, `avg_yearbuilt`, `building_count`, `total_bldg_area`

**Output file:** `csv/Houston/03_building_characteristics.csv` or `csv/NYC/03_building_characteristics.csv`

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.neighbors import BallTree
import warnings
warnings.filterwarnings('ignore')

# ── Load config ────────────────────────────────────────
GRID_CONFIG = "grid.json"
with open(GRID_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

CELL_SIZE_M = config["grid_cell_size_m"]
CSV_DIR = config.get("csv_dir", "csv")
os.makedirs(CSV_DIR, exist_ok=True)

# Determine which data source to use
NEEDS_HCAD = config["feature_flags"].get("needs_hcad", False)
NEEDS_PLUTO = config["feature_flags"].get("needs_pluto", False)

if not NEEDS_HCAD and not NEEDS_PLUTO:
    print("Neither HCAD nor PLUTO enabled — skipping notebook 03.")
    df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
    df_empty = pd.DataFrame({"cell_id": df_grid["cell_id"]})
    for col in ["avg_floors", "avg_yearbuilt", "building_count", "total_bldg_area"]:
        df_empty[col] = np.nan
    df_empty.to_csv(f"{CSV_DIR}/03_building_characteristics.csv", index=False)
    raise SystemExit("Skipped — needs_hcad=false and needs_pluto=false")

print(f"Config loaded:")
print(f"  needs_hcad: {NEEDS_HCAD}")
print(f"  needs_pluto: {NEEDS_PLUTO}")
print(f"  csv_dir: {CSV_DIR}")

## HCAD (Houston)

Load and aggregate HCAD building data.

In [ ]:
df_data = None

if NEEDS_HCAD:
    print("\n" + "="*60)
    print("HCAD (Houston)")
    print("="*60)
    
    HCAD_PATH = config["hcad_path"]
    COLS = ["latitude", "longitude", "numfloors", "yearbuilt", "bldgarea", "numbldgs", "lotarea"]
    
    print(f"\nLoading HCAD from {HCAD_PATH}...")
    df_data = pd.read_csv(HCAD_PATH, usecols=COLS)
    
    # Convert to numeric
    for col in ["numfloors", "yearbuilt", "bldgarea", "numbldgs", "lotarea"]:
        df_data[col] = pd.to_numeric(df_data[col], errors="coerce")
    
    df_data["latitude"] = pd.to_numeric(df_data["latitude"], errors="coerce")
    df_data["longitude"] = pd.to_numeric(df_data["longitude"], errors="coerce")
    df_data = df_data.dropna(subset=["latitude", "longitude"]).copy()
    
    # Clean floor counts and years
    df_data.loc[df_data["yearbuilt"] < 1700, "yearbuilt"] = np.nan
    df_data.loc[df_data["numfloors"] <= 0, "numfloors"] = np.nan
    
    print(f"✓ Loaded {len(df_data):,} records")
    print(f"  Lat: {df_data['latitude'].min():.4f} to {df_data['latitude'].max():.4f}")
    print(f"  Lon: {df_data['longitude'].min():.4f} to {df_data['longitude'].max():.4f}")
else:
    print("\nHCAD disabled (needs_hcad=false) — skipping HCAD section")

## PLUTO (NYC)

Load and aggregate PLUTO building data.

In [ ]:
if NEEDS_PLUTO:
    print("\n" + "="*60)
    print("PLUTO (NYC)")
    print("="*60)
    
    PLUTO_PATH = config["pluto_path"]
    BOROUGH_CODES = config["borough_codes"]
    BOROUGH_FILTER = config["borough_filter"]
    boro_code_filter = [str(BOROUGH_CODES[b]) for b in BOROUGH_FILTER]
    
    COLS = ["borocode", "numfloors", "yearbuilt", "lotarea", "bldgarea", "numbldgs",
            "latitude", "longitude"]
    
    print(f"\nLoading PLUTO from {PLUTO_PATH}...")
    df_data = pd.read_csv(PLUTO_PATH, usecols=COLS)
    df_data = df_data[df_data["borocode"].astype(str).isin(boro_code_filter)].copy()
    
    # Convert to numeric
    for col in ["numfloors", "yearbuilt", "lotarea", "bldgarea", "numbldgs"]:
        df_data[col] = pd.to_numeric(df_data[col], errors="coerce")
    df_data["latitude"] = pd.to_numeric(df_data["latitude"], errors="coerce")
    df_data["longitude"] = pd.to_numeric(df_data["longitude"], errors="coerce")
    df_data = df_data.dropna(subset=["latitude", "longitude"]).copy()
    
    # Clean
    df_data.loc[df_data["yearbuilt"] < 1700, "yearbuilt"] = np.nan
    df_data.loc[df_data["numfloors"] <= 0, "numfloors"] = np.nan
    
    print(f"✓ Loaded {len(df_data):,} records (filtered to {', '.join(BOROUGH_FILTER)})")
    print(f"  Lat: {df_data['latitude'].min():.4f} to {df_data['latitude'].max():.4f}")
    print(f"  Lon: {df_data['longitude'].min():.4f} to {df_data['longitude'].max():.4f}")
else:
    print("\nPLUTO disabled (needs_pluto=false) — skipping PLUTO section")

## Spatial Assignment

Assign properties to grid cells using BallTree (same method as notebook 01).

In [ ]:
if df_data is not None:
    print("\n" + "="*60)
    print("SPATIAL ASSIGNMENT")
    print("="*60)
    
    # Load grid from notebook 01
    df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
    print(f"\n✓ Loaded {len(df_grid):,} grid cells from 01_grid_definition.csv")
    print(f"  Sample cell IDs: {df_grid['cell_id'].head().tolist()}")
    
    # Use BallTree for consistent spatial assignment (matching notebook 01)
    print(f"\nAssigning {len(df_data):,} properties to nearest grid cells...")
    
    coords = df_grid[["cell_lat", "cell_lon"]].values
    tree = BallTree(np.radians(coords), metric="haversine")
    
    data_coords = df_data[["latitude", "longitude"]].values
    distances, indices = tree.query(np.radians(data_coords), k=1)
    
    df_data["cell_id"] = df_grid.iloc[indices.flatten()]["cell_id"].values
    
    # Check assignment distribution
    assignment_counts = df_data["cell_id"].value_counts()
    print(f"\n✓ Assigned {len(df_data):,} records to {len(assignment_counts):,} grid cells")
    print(f"  Per-cell stats:")
    print(f"    Min: {assignment_counts.min():,}")
    print(f"    Max: {assignment_counts.max():,}")
    print(f"    Mean: {assignment_counts.mean():.0f}")
    print(f"    Median: {assignment_counts.median():.0f}")
    print(f"  Distance stats (km):")
    print(f"    Min: {distances.min()*6371:.3f}")
    print(f"    Max: {distances.max()*6371:.3f}")
    print(f"    Mean: {distances.mean()*6371:.3f}")
else:
    print("\nNo data loaded — skipping spatial assignment")

## Aggregation

Aggregate building characteristics per grid cell.

In [ ]:
if df_data is not None:
    print("\n" + "="*60)
    print("AGGREGATION BY CELL")
    print("="*60)
    
    # Aggregate per grid cell
    agg = df_data.groupby("cell_id").agg(
        avg_floors=("numfloors", "mean"),
        avg_yearbuilt=("yearbuilt", "mean"),
        total_bldg_area=("bldgarea", "sum"),
        building_count=("numbldgs", "sum"),
    ).reset_index()
    
    # Format columns
    agg["avg_floors"] = agg["avg_floors"].round(1)
    agg["avg_yearbuilt"] = agg["avg_yearbuilt"].round(0).astype("Int64")
    agg["total_bldg_area"] = agg["total_bldg_area"].round(0).astype("Int64")
    agg["building_count"] = agg["building_count"].astype("Int64")
    
    # Ensure all grid cells are present (with NaN for empty cells)
    df_result = df_grid[["cell_id"]].merge(agg, on="cell_id", how="left")
    
    print(f"\n✓ Aggregated {len(df_grid):,} grid cells")
    print(f"  Cells with data: {agg.shape[0]:,}")
    print(f"  Cells with no data (NaN): {(df_result['avg_floors'].isna().sum()):,}")
    
    print(f"\nStatistics:")
    print(df_result.describe().round(1).to_string())
else:
    print("\nNo data to aggregate")

## Save Output

In [ ]:
if df_data is not None:
    output_path = f"{CSV_DIR}/03_building_characteristics.csv"
    df_result.to_csv(output_path, index=False, encoding="utf-8")
    
    print(f"\n✓ Saved: {output_path}")
    print(f"  {len(df_result):,} rows × {df_result.shape[1]} columns")
    print(f"\nFirst 10 rows:")
    print(df_result.head(10).to_string())
else:
    print("\nNo output to save")